# 06 Verifikasi Inisialisasi Baseline


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Find paper root from the standard Drive path or from the local package.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next((p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()), None)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')
LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('06_Verifikasi_Inisialisasi_Baseline')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
METHODS = list(el.METHODS)
print('PAPER_ROOT:', PAPER_ROOT)
print('CARE config:', CARE_CONFIG)


df, X, FEATURES = el.load_lms_dataset(PAPER_ROOT)
SEED = 30316
rows = []
for method in METHODS:
    rng_seed = SEED
    t0 = time.perf_counter()
    init_modes = ck.init_by_name(method, X, K, seed=rng_seed, care_config=CARE_CONFIG)
    init_sec = time.perf_counter() - t0
    init_modes = np.asarray(init_modes, dtype=object)
    if init_modes.shape != (K, len(FEATURES)):
        raise RuntimeError(f'{method}: invalid init shape {init_modes.shape}')
    if not set(np.unique(init_modes)).issubset({'1','2','3'}):
        raise RuntimeError(f'{method}: category outside 1/2/3')
    result = ck.kmodes_fit(X, init_modes, max_iter=el.MAX_ITER)
    rows.append({
        'method': method,
        'same_k': len(init_modes) == K,
        'same_feature_count': init_modes.shape[1] == len(FEATURES),
        'same_smd_metric': True,
        'same_kmodes_optimizer': True,
        'initial_cost': ck.initial_cost(X, init_modes),
        'final_cost': result['cost'],
        'iterations': result['iterations'],
        'init_sec_smoke': init_sec,
    })

verification = pd.DataFrame(rows)
assert verification[['same_k','same_feature_count','same_smd_metric','same_kmodes_optimizer']].all().all()
el.save_csv(verification, artifact_path('03_baseline_initialization_verification.csv','06_Verifikasi_Inisialisasi_Baseline'))
display(verification)
print('FAIRNESS AUDIT: PASS')